In [21]:
""" Create the observational data set after controlling for pretreatment effects
    "extract_obs_productivity.ipynb" only applies it on biomass

    Because of the considerable pre-treatment effect reported in Hanson et al. 2025,
    we also remove the pre-treatment effect from AGNPP of the trees

    Hanson, P. J., Griffiths, N. A., Salmon, V. G., Birkebak, J. M., Warren, J. M., Phillips, J. R., et al. (2025). Peatland Plant Community Changes in Annual Production and Composition Through 8 Years of Warming Manipulations Under Ambient and Elevated CO2 Atmospheres. Journal of Geophysical Research: Biogeosciences, 130(2), e2024JG008511. https://doi.org/10.1029/2024JG008511
"""
import pandas as pd
import os
import numpy as np
import statsmodels.formula.api as smf

In [22]:
def collect_data():
    obs_paul = pd.read_excel(os.path.join(os.environ['HOME'],
        'Git', 'phenology_elm', "SPRUCE C Budget Summary 28Apr2022EXP.xlsx"),
        sheet_name="DataForPythonRead", skiprows=1,  engine="openpyxl")
    obs_paul = obs_paul.loc[obs_paul["Year"] != 2020, :]
    obs_paul = obs_paul.set_index(['Plot', 'Year']).sort_index()

    obs_verity = pd.read_csv(os.path.join(os.environ['HOME'],
        'Git', 'phenology_elm', 'SalmonSPRUCE_2016to2021_AbovegroundPFT_CNPbudget_20240208.csv'))
    # match by plot and year to temperature
    obs_verity['Plot'] = [f'P{p:02d}' for p in obs_verity['Plot']]
    obs_verity = obs_verity.set_index(['Plot', 'Year', 'PFT']).unstack()
    obs_verity = obs_verity.loc[:, (slice(None), 
                                    ['Sphagnum', 'evergreen conifer', 'deciduous conifer',  
                                        'shrub'])]
    obs_verity2 = obs_verity.loc[:, ['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear',
                                    'Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear']]
    obs_data = pd.concat([obs_paul, obs_verity2], axis = 1, join = 'outer')
    obs_data = obs_data.reset_index()

    # Merge 0 and Amb
    obs_data['eCO2'] = np.where(obs_data['CO2'].isna(), np.nan, obs_data['CO2'] == 500)

    # remove unecessary variables
    obs_data.drop(['CO2', 'Temperature', 'ANPP Tree (~48%C)', 
                   'ANPP Shrub (~50%C)', 'CH4', 'NCE', 'DOC Loss', 
                   ('ABGbiomass_gCperm2', 'Sphagnum'), 
                   ('ABGnpp_gCperm2peryear', 'Sphagnum'), 
                   ('Pretrt_ABGbiomass_gCperm2', 'Sphagnum'),
                   ('Pretrt_ABGnpp_gCperm2peryear', 'Sphagnum')], 
                  axis = 1, inplace = True)

    return obs_data

obs_data = collect_data()

In [23]:
obs_data2 = pd.DataFrame(np.nan, 
    index = pd.MultiIndex.from_frame(obs_data[['Plot','Year']]),
    columns = ['Tair', 
               'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub', # gC m-2
               'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss', # gC m-2 yr-1
               'BGNPP_TreeShrub', # fine root NPP, gC m-2 yr-1
               'AGNPP', # tree + shrub + moss
               'HR']
)

In [24]:
obs_data2.loc[:, 'Tair'] = obs_data.loc[:, 'Mean Annual Temp. at 2 m'].values
obs_data2.loc[:, 'BGNPP_TreeShrub'] = obs_data['BNPP Tree & Shrub'].values
obs_data2.loc[:, 'HR'] = obs_data.loc[:, 'RHCO2'].values
obs_data2.loc[:, 'NPP_moss'] = obs_data.loc[:, 'NPP Sphag.'].values

In [30]:
# Use linear regression to remove pre-treatment effect on the
# aboveground biomass & aboveground NPP of trees
# 
# Post treatment level ~ I(eCO2) + Tair + Pre-treatment level
#    + I(eCO2) x Tair + I(eCO2) x Pre-treatment level
#    + Tair x Pre-treatment level
#
# Subtract out
#    Pre-treatment level + I(eCO2) x Pre-treatment level
#    + Tair x Pre-treatment level

def create_X(obs_data, varpre, varname, pft2):
    temp_df = {
        'eCO2': obs_data['eCO2'],
        'Tair': obs_data['Mean Annual Temp. at 2 m'].values,
        'Pretrt': obs_data[(varpre,pft2)].values,
        'Postrt': obs_data[(varname,pft2)].values,
    }
    temp_df = pd.DataFrame(temp_df)
    temp_df = temp_df.dropna(how = 'any', axis = 0)

    return temp_df


for varname, varpre, varfinal in \
    zip(['ABGbiomass_gCperm2', 'ABGnpp_gCperm2peryear'], 
        ['Pretrt_ABGbiomass_gCperm2', 'Pretrt_ABGnpp_gCperm2peryear'],
        ['AGBiomass', 'AGNPP']):

    for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                        ['evergreen conifer','deciduous conifer','shrub']):

        # Form the dataset
        df = create_X(obs_data, varpre, varname, pft2)

        # Full model
        full_formula = ("Postrt ~ eCO2 + Tair + Pretrt + eCO2:Tair + eCO2:Pretrt + Tair:Pretrt")
        full_mod = smf.ols(full_formula, data=df).fit()

        # Drop insignificant terms and refit
        keepers = [
            term for term, p in full_mod.pvalues.items()
            if (term == "Intercept") or (p <= 0.05)
        ]

        if not ('Pretrt' in keepers or 'eCO2:Pretrt' in keepers):
            obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values

            val0 = np.nanmean(obs_data[(varname, pft2)].values)
            se0 = np.nanstd(obs_data[(varname, pft2)].values)
            val1 = val0
            se1 = se0

            print(varname, pft, f"eCO2=0 : {val0:.4g} ± {se0:.4g}", f"eCO2=1 : {val1:.4g} ± {se1:.4g}")
            print()

        else:
            reduced_formula = "Postrt ~ " + " + ".join(
                k for k in keepers if k != "Intercept"
            )
            results = smf.ols(reduced_formula, data=df).fit()

            # Calculate the pretreatment effect (intercept + pretreatment)
            # and uncertainty estimation of this pretreatment effect

            # helper for value ± SE of any linear combo -------------------------------
            def combo(weights):               # weights dict {term: value}
                beta = results.params         # regression coefficient vector
                V    = results.cov_params()   # variance–covariance matrix of regression coefficient
                vec = np.array([weights.get(name, 0.0) for name in beta.index])
                val = vec @ beta
                se  = np.sqrt(vec @ V.values @ vec)
                return val, se

            # Because we want to estimate the intercept at temperature = 0
            # This is wrong!!! We cannot have temperature = 0

            # eCO2 = 0 ---------------------------------------------------------------
            weights = {'Intercept': 1}
            if 'Pretrt' in results.pvalues.index and results.pvalues['Pretrt'] <= 0.05:
                weights['Pretrt'] = df['Pretrt'].mean()
            val0, se0 = combo(weights)

            # eCO2 = 1 ---------------------------------------------------------------
            if 'eCO2:Pretrt' in results.pvalues.index and results.pvalues['eCO2:Pretrt'] <= 0.05:
                weights['eCO2:Pretrt'] = df['Pretrt'].mean()
            val1, se1 = combo(weights)

            # Print the regression outcomes
            #print('--------------------', varname, pft, '--------------------')
            #print(results.summary()) # too much information
            print(varname, pft, f"eCO2=0 : {val0:.4g} ± {se0:.4g}", f"eCO2=1 : {val1:.4g} ± {se1:.4g}")
            print()

            # Subtract the pretreatment effect if significant
            # (add back the mean of the pretreatment level)
            obs_data2.loc[:, f'{varfinal}_{pft}'] = obs_data[(varname, pft2)].values
            if 'Pretrt' in results.pvalues.index and results.pvalues['Pretrt'] <= 0.05:       
                obs_data2.loc[:, f'{varfinal}_{pft}'] -= \
                    results.params['Pretrt'] * (obs_data[(varpre, pft2)].values - \
                                                obs_data[(varpre, pft2)].mean())
            if 'eCO2:Pretrt' in results.pvalues.index and results.pvalues['eCO2:Pretrt'] <= 0.05:
                obs_data2.loc[:, f'{varfinal}_{pft}'] -= \
                    results.params['eCO2:Pretrt'] * obs_data['eCO2'].values * \
                        (obs_data[(varpre, pft2)].values - obs_data[(varpre, pft2)].mean())

ABGbiomass_gCperm2 Spruce eCO2=0 : 1039 ± 29.48 eCO2=1 : 1039 ± 29.48

ABGbiomass_gCperm2 Tamarack eCO2=0 : 260.8 ± 22.56 eCO2=1 : 260.8 ± 22.56

ABGbiomass_gCperm2 Shrub eCO2=0 : 286.2 ± 136.7 eCO2=1 : 286.2 ± 136.7

ABGnpp_gCperm2peryear Spruce eCO2=0 : 141.1 ± 16.69 eCO2=1 : 110.3 ± 16.47

ABGnpp_gCperm2peryear Tamarack eCO2=0 : 37.75 ± 8.05 eCO2=1 : 37.75 ± 8.05

ABGnpp_gCperm2peryear Shrub eCO2=0 : 97.68 ± 44.58 eCO2=1 : 97.68 ± 44.58



In [26]:
"""for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], 
                    ['evergreen conifer','deciduous conifer','shrub']):
    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] = \
        obs_data[('ABGnpp_gCperm2peryear', pft2)].values / \
        obs_data[('ABGbiomass_gCperm2', pft2)].values

obs_data2.loc[:, 'BGtoAG_TreeShrub'] = \
    obs_data2.loc[:, 'BGNPP_TreeShrub'].values / \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub']).values"""

"for pft,pft2 in zip(['Spruce', 'Tamarack', 'Shrub'], \n                    ['evergreen conifer','deciduous conifer','shrub']):\n    obs_data2.loc[:, f'AGNPPtoBiomass_{pft}'] =         obs_data[('ABGnpp_gCperm2peryear', pft2)].values /         obs_data[('ABGbiomass_gCperm2', pft2)].values\n\nobs_data2.loc[:, 'BGtoAG_TreeShrub'] =     obs_data2.loc[:, 'BGNPP_TreeShrub'].values /     (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] +      obs_data2.loc[:, 'AGNPP_Shrub']).values"

In [27]:
obs_data2.loc[:, 'NPP'] = \
    (obs_data2.loc[:, 'AGNPP_Spruce'] + obs_data2.loc[:, 'AGNPP_Tamarack'] + \
     obs_data2.loc[:, 'AGNPP_Shrub'] + obs_data2.loc[:, 'BGNPP_TreeShrub'] + \
     obs_data2.loc[:, 'NPP_moss']).values

In [28]:
path_out = os.path.join(os.environ['PROJDIR'], 'ELM_Phenology', 'output', 'extract')
obs_data2.to_csv(os.path.join(path_out, 'extract_obs_productivity.csv'))

obs_data2

Tair  AGBiomass_Spruce  AGBiomass_Tamarack  AGBiomass_Shrub  \
Plot Year                                                                     
P04  2016  12.700000        971.605207          218.056742          354.660   
     2017  11.300000       1039.226207          235.789742          418.203   
     2018  10.800000       1128.137207          251.729742          273.278   
     2019   9.967064       1189.958207          275.097742          311.169   
     2021  12.100000       1260.286207          262.508742          423.661   
...              ...               ...                 ...              ...   
P13  2020        NaN       1091.384107          330.092073              NaN   
P16  2020        NaN       1051.702447          149.777698              NaN   
P17  2020        NaN        855.141240          242.480901              NaN   
P19  2020        NaN       1046.081468          307.046608              NaN   
P20  2020        NaN       1354.180280          234.037479              NaN   

           AGNPP_Spruce  AGNPP_Tamarack  AGNPP_Shrub  NPP_moss  \
Plot Year                                                        
P04  2016    110.260242       -7.868811      177.250   105.600   
     2017     87.727242       13.371189      163.414    23.900   
     2018    128.580242       23.258189       93.889    29.100   
     2019    117.819242       31.609189      126.523    16.591   
     2021    131.924242       16.035189      150.396     9.600   
...                 ...             ...          ...       ...   
P13  2020           NaN       50.333860          NaN       NaN   
P16  2020           NaN       34.839715          NaN       NaN   
P17  2020           NaN       24.628174          NaN       NaN   
P19  2020           NaN       49.672378          NaN       NaN   
P20  2020           NaN       35.722925          NaN       NaN   

           BGNPP_TreeShrub  AGNPP         HR         NPP  
Plot Year                                                 
P04  2016        104.90000    NaN -537.30000  490.141431  
     2017        108.80000    NaN -453.30000  397.212431  
     2018        106.85000    NaN -456.80000  381.677431  
     2019        107.82500    NaN -381.07052  400.367431  
     2021        107.58125    NaN -453.75570  415.536681  
...                    ...    ...        ...         ...  
P13  2020              NaN    NaN        NaN         NaN  
P16  2020              NaN    NaN        NaN         NaN  
P17  2020              NaN    NaN        NaN         NaN  
P19  2020              NaN    NaN        NaN         NaN  
P20  2020              NaN    NaN        NaN         NaN  

[70 rows x 12 columns]

In [29]:
obs_data2.columns

Index(['Tair', 'AGBiomass_Spruce', 'AGBiomass_Tamarack', 'AGBiomass_Shrub',
       'AGNPP_Spruce', 'AGNPP_Tamarack', 'AGNPP_Shrub', 'NPP_moss',
       'BGNPP_TreeShrub', 'AGNPP', 'HR', 'NPP'],
      dtype='object')